## Day 1: Setup and Foundation

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Check if the file exists in your specified Drive path
file_path = '/content/drive/MyDrive/ecommerce_rfm_project/online_retail_II.xlsx'

print("File exists:", os.path.exists(file_path))

File exists: True


## Day 2: Data Ingestion and Inspecting Schemas

In [3]:
import pandas as pd

In [4]:
excel_workbook = pd.ExcelFile(file_path)
excel_workbook.sheet_names


['Year 2009-2010', 'Year 2010-2011']

In [5]:
df_temp = pd.read_excel(excel_workbook, sheet_name='Year 2009-2010')
df_temp.to_csv('year_2009_2010.csv', index=False)

In [6]:
del df_temp

In [7]:
import gc

gc.collect()

525511

In [8]:
df_temp2 = pd.read_excel(excel_workbook, sheet_name='Year 2010-2011')
df_temp2.to_csv('year_2010_2011.csv', index=False)

In [9]:
import gc

del df_temp2
gc.collect()

541960

In [10]:
df_temp1 = pd.read_csv('year_2009_2010.csv')
df_temp2 = pd.read_csv('year_2010_2011.csv')

In [11]:
df_temp1_columns = df_temp1.columns.str.strip().str.lower()
df_temp2_columns = df_temp2.columns.str.strip().str.lower()
print(df_temp1_columns)
print(df_temp2_columns)


Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer id', 'country'],
      dtype='object')
Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer id', 'country'],
      dtype='object')


In [12]:
df_temp1.columns = df_temp1.columns.str.strip().str.lower()
df_temp2.columns = df_temp2.columns.str.strip().str.lower()

In [13]:
print(df_temp1.dtypes)
print(df_temp2.dtypes)

invoice         object
stockcode       object
description     object
quantity         int64
invoicedate     object
price          float64
customer id    float64
country         object
dtype: object
invoice         object
stockcode       object
description     object
quantity         int64
invoicedate     object
price          float64
customer id    float64
country         object
dtype: object


In [14]:
print(df_temp1.isnull().sum())
print(df_temp2.isnull().sum())

invoice             0
stockcode           0
description      2928
quantity            0
invoicedate         0
price               0
customer id    107927
country             0
dtype: int64
invoice             0
stockcode           0
description      1454
quantity            0
invoicedate         0
price               0
customer id    135080
country             0
dtype: int64


In [15]:
total_rows1 = len(df_temp1)
missing_percentage = (df_temp1.isnull().sum() / total_rows1) * 100
print(missing_percentage)

invoice         0.000000
stockcode       0.000000
description     0.557225
quantity        0.000000
invoicedate     0.000000
price           0.000000
customer id    20.539488
country         0.000000
dtype: float64


In [16]:
total_rows2 = len(df_temp2)
missing_percentage = (df_temp2.isnull().sum() / total_rows2) * 100
print(missing_percentage)

invoice         0.000000
stockcode       0.000000
description     0.268310
quantity        0.000000
invoicedate     0.000000
price           0.000000
customer id    24.926648
country         0.000000
dtype: float64


In [17]:
print(df_temp1.describe())
print(df_temp2.describe())

            quantity          price    customer id
count  525461.000000  525461.000000  417534.000000
mean       10.337667       4.688834   15360.645478
std       107.424110     146.126914    1680.811316
min     -9600.000000  -53594.360000   12346.000000
25%         1.000000       1.250000   13983.000000
50%         3.000000       2.100000   15311.000000
75%        10.000000       4.210000   16799.000000
max     19152.000000   25111.090000   18287.000000
            quantity          price    customer id
count  541910.000000  541910.000000  406830.000000
mean        9.552234       4.611138   15287.684160
std       218.080957      96.759765    1713.603074
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


## Day 3: Data Quality Audit and Cleaning Pipeline

In [18]:
combined_df = pd.concat([df_temp1, df_temp2], axis=0)
display(combined_df)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [19]:
filtered_df = combined_df[combined_df['invoice'].str.contains('C', na=False)]
display(filtered_df)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [20]:
(filtered_df['quantity'] <= 0).all()

np.False_

In [21]:
len(combined_df [combined_df['quantity'] < 0])

22950

In [22]:
filtered_df2 = combined_df[~combined_df['invoice'].str.contains('C', na=False)]
display(filtered_df2)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [23]:
negative_quantities = filtered_df2 [filtered_df2['quantity'] < 0]
display(negative_quantities)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom
536910,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom


In [24]:
len(combined_df [(combined_df['price'] <= 0) & (combined_df['quantity']>0)])

2750

In [25]:
filtered_df3 = combined_df [(combined_df['price'] <= 0) & (combined_df['quantity']>0)]
display(filtered_df3)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535334,581211,22142,check,14,2011-12-07 18:36:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom


In [26]:
clean_df = combined_df [(combined_df ['quantity'] > 0) & (combined_df ['price'] > 0) & (combined_df ['customer id'].notnull())]
display(clean_df)


,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [27]:
clean_df = clean_df.drop_duplicates()
display(clean_df)


,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [28]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   invoice      779425 non-null  object 
 1   stockcode    779425 non-null  object 
 2   description  779425 non-null  object 
 3   quantity     779425 non-null  int64  
 4   invoicedate  779425 non-null  object 
 5   price        779425 non-null  float64
 6   customer id  779425 non-null  float64
 7   country      779425 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 53.5+ MB


In [29]:
type(clean_df['invoicedate'].iloc[-1])

str

In [30]:
clean_df = combined_df[(combined_df['quantity'] > 0) &
                       (combined_df['price'] > 0) &
                       (combined_df['customer id'].notnull())].copy()

clean_df = clean_df.drop_duplicates()

In [31]:
clean_df['invoicedate'] = pd.to_datetime(clean_df['invoicedate'])

In [32]:
clean_df['customer id'] = clean_df['customer id'].astype('int64')


In [33]:
clean_df['total_amount'] = clean_df['quantity'] * clean_df['price']

In [34]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 541909
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   invoice       779425 non-null  object        
 1   stockcode     779425 non-null  object        
 2   description   779425 non-null  object        
 3   quantity      779425 non-null  int64         
 4   invoicedate   779425 non-null  datetime64[ns]
 5   price         779425 non-null  float64       
 6   customer id   779425 non-null  int64         
 7   country       779425 non-null  object        
 8   total_amount  779425 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 59.5+ MB


In [35]:
clean_df['total_amount'].describe()

,total_amount
count,779425.000000
mean,22.291823
std,227.427075
min,0.001000
25%,4.950000
50%,12.480000
75%,19.800000
max,168469.600000


In [36]:
clean_df.shape


(779425, 9)

## Day 4: Exploratory Data Analysis (EDA)


In [37]:
min_date = clean_df['invoicedate'].min()
max_date = clean_df['invoicedate'].max()
print(f"Minimum Date: {min_date}")
print(f"Maximum Date: {max_date}")

Minimum Date: 2009-12-01 07:45:00
Maximum Date: 2011-12-09 12:50:00


In [38]:
unique_ids = clean_df['customer id'].nunique()
print(f"Number of Unique Customer IDs: {unique_ids}")

Number of Unique Customer IDs: 5878


In [39]:
unique_invoices = clean_df['invoice'].nunique()
print(f"Number of Unique Invoices: {unique_invoices}")

Number of Unique Invoices: 36969


In [40]:
total_revenue = clean_df['total_amount'].sum()
print(f"Total Revenue: ${total_revenue:.2f}")


Total Revenue: $17374804.27


In [41]:
country_transactions = clean_df['country'].value_counts()
print(country_transactions)


country
United Kingdom          700388
Germany                  16432
EIRE                     15565
France                   13511
Netherlands               5085
Spain                     3662
Belgium                   3055
Switzerland               3005
Portugal                  2356
Australia                 1789
Channel Islands           1551
Italy                     1442
Sweden                    1317
Norway                    1289
Cyprus                    1136
Finland                   1032
Austria                    922
Denmark                    778
Greece                     657
Unspecified                518
Poland                     504
Japan                      468
USA                        409
United Arab Emirates       383
Singapore                  339
Israel                     319
Malta                      282
Canada                     228
Iceland                    222
Lithuania                  154
RSA                        122
Brazil                      94


In [42]:
rev_per_country = clean_df.groupby('country')['total_amount'].sum().sort_values(ascending=False)
print(rev_per_country)


country
United Kingdom          1.438923e+07
EIRE                    6.165705e+05
Netherlands             5.540381e+05
Germany                 4.250197e+05
France                  3.487690e+05
Australia               1.692835e+05
Spain                   1.083325e+05
Switzerland             1.000619e+05
Sweden                  9.151582e+04
Denmark                 6.858069e+04
Belgium                 6.538782e+04
Norway                  5.632250e+04
Portugal                5.555478e+04
Channel Islands         4.462333e+04
Japan                   4.302391e+04
Italy                   3.210817e+04
Finland                 2.992554e+04
Singapore               2.531706e+04
Cyprus                  2.484995e+04
Austria                 2.361301e+04
Greece                  1.909619e+04
Poland                  1.065429e+04
Israel                  1.041524e+04
United Arab Emirates    9.202690e+03
Unspecified             8.607350e+03
USA                     8.366860e+03
Malta                   8.0990

In [43]:
rev_per_customer = clean_df.groupby('customer id')['total_amount'].sum().sort_values(ascending=False)
print(rev_per_customer)

quantity_per_customer = clean_df.groupby('customer id')['quantity'].sum().sort_values(ascending=False)
print(quantity_per_customer)


customer id
18102    580987.04
14646    528602.52
14156    313437.62
14911    291420.81
17450    244784.25
           ...    
15913         6.30
14792         6.20
16738         3.75
13788         3.75
14095         2.95
Name: total_amount, Length: 5878, dtype: float64
customer id
14646    367193
13902    220600
13694    188201
18102    181645
14156    164325
          ...  
12570         1
17378         1
12846         1
13290         1
14328         1
Name: quantity, Length: 5878, dtype: int64


In [44]:
rev_per_customer.describe()

,total_amount
count,5878.000000
mean,2955.904095
std,14440.852688
min,2.950000
25%,342.280000
50%,867.740000
75%,2248.305000
max,580987.040000


In [45]:
clean_df['year_month'] = clean_df['invoicedate'].dt.to_period('M')
rev_per_month = clean_df.groupby('year_month')['total_amount'].sum()
print(rev_per_month)

year_month
2009-12     683504.010
2010-01     555802.672
2010-02     504558.956
2010-03     696978.471
2010-04     591982.002
2010-05     597833.380
2010-06     636371.130
2010-07     589736.170
2010-08     602224.600
2010-09     829013.951
2010-10    1033112.010
2010-11    1166460.022
2010-12     570422.730
2011-01     568101.310
2011-02     446084.920
2011-03     594081.760
2011-04     468374.331
2011-05     677355.150
2011-06     660046.050
2011-07     598962.901
2011-08     644051.040
2011-09     950690.202
2011-10    1035642.450
2011-11    1156205.610
2011-12     517208.440
Freq: M, Name: total_amount, dtype: float64


In [46]:
clean_df['day_of_week'] = clean_df['invoicedate'].dt.day_name()
rev_per_day = clean_df.groupby('day_of_week')['total_amount'].sum()
print(rev_per_day)

day_of_week
Friday       2728473.173
Monday       2778201.566
Saturday        9803.050
Sunday       1768669.052
Thursday     3745783.432
Tuesday      3322830.142
Wednesday    3021043.853
Name: total_amount, dtype: float64


In [47]:
clean_df['hour_of_day'] = clean_df['invoicedate'].dt.hour
rev_per_hour = clean_df.groupby('hour_of_day')['total_amount'].sum()
print(rev_per_hour)

hour_of_day
6           4.250
7       75745.570
8      524612.550
9     1484450.541
10    2320711.443
11    2235870.393
12    2690583.501
13    2347078.714
14    1962016.161
15    1839607.762
16    1055274.691
17     484477.632
18     190860.290
19     124825.770
20      38685.000
Name: total_amount, dtype: float64


In [48]:
last_date = clean_df['invoicedate'].max()
print(f"Last Transaction Date: {last_date}")

Last Transaction Date: 2011-12-09 12:50:00


In [49]:
reference_date = last_date + pd.DateOffset(days=1)
print(f"Reference Date: {reference_date}")

Reference Date: 2011-12-10 12:50:00


In [50]:
aggregate = clean_df.groupby('customer id').agg({
    'invoicedate':'max',
    'invoice': 'nunique',
    'total_amount': 'sum'

})

aggregate['Recency'] = (reference_date - aggregate['invoicedate']).dt.days
aggregate['Frequency'] = aggregate['invoice']
aggregate['Monetary'] = aggregate['total_amount']
aggregate = aggregate.drop(columns=['invoicedate', 'invoice', 'total_amount'])
print(aggregate)

             Recency  Frequency  Monetary
customer id                              
12346            326         12  77556.46
12347              2          8   4921.53
12348             75          5   2019.40
12349             19          4   4428.69
12350            310          1    334.40
...              ...        ...       ...
18283              4         22   2664.90
18284            432          1    461.68
18285            661          1    427.00
18286            477          2   1296.43
18287             43          7   4182.99

[5878 rows x 3 columns]


In [51]:
aggregate.describe()

,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


In [52]:
labels = ['4', '3', '2', '1']
aggregate['R_score'] = pd.qcut(aggregate['Recency'], q=4, labels=labels)

labels = ['1', '2', '3', '4']
aggregate['F_score'] = pd.cut(aggregate['Frequency'], labels=labels, bins = [0, 1, 3, 7, float('inf')])

labels = ['1', '2', '3', '4']
aggregate['M_score'] = pd.qcut(aggregate['Monetary'], q=4, labels=labels)


In [53]:
aggregate['R_score'].value_counts().sort_index()


,count
R_score,
4,1510
3,1443
2,1466
1,1459


In [54]:
aggregate['F_score'].value_counts().sort_index()

,count
F_score,
1,1623
2,1608
3,1348
4,1299


In [55]:
aggregate['M_score'].value_counts().sort_index()

,count
M_score,
1,1470
2,1469
3,1469
4,1470


In [56]:
aggregate['RFM_segment'] = aggregate['R_score'].astype(str) + aggregate['F_score'].astype(str) + aggregate['M_score'].astype(str)
print(aggregate)

             Recency  Frequency  Monetary R_score F_score M_score RFM_segment
customer id                                                                  
12346            326         12  77556.46       2       4       4         244
12347              2          8   4921.53       4       4       4         444
12348             75          5   2019.40       3       3       3         333
12349             19          4   4428.69       4       3       4         434
12350            310          1    334.40       2       1       1         211
...              ...        ...       ...     ...     ...     ...         ...
18283              4         22   2664.90       4       4       4         444
18284            432          1    461.68       1       1       2         112
18285            661          1    427.00       1       1       2         112
18286            477          2   1296.43       1       2       3         123
18287             43          7   4182.99       3       3       

In [57]:
aggregate['RFM_score'] = aggregate['R_score'].astype(int) + aggregate['F_score'].astype(int) + aggregate['M_score'].astype(int)
print(aggregate)

             Recency  Frequency  Monetary R_score F_score M_score RFM_segment  \
customer id                                                                     
12346            326         12  77556.46       2       4       4         244   
12347              2          8   4921.53       4       4       4         444   
12348             75          5   2019.40       3       3       3         333   
12349             19          4   4428.69       4       3       4         434   
12350            310          1    334.40       2       1       1         211   
...              ...        ...       ...     ...     ...     ...         ...   
18283              4         22   2664.90       4       4       4         444   
18284            432          1    461.68       1       1       2         112   
18285            661          1    427.00       1       1       2         112   
18286            477          2   1296.43       1       2       3         123   
18287             43        

In [58]:
aggregate['RFM_segment'].nunique()

62

In [59]:
count = (aggregate['RFM_segment'] == '444').sum()
print(count)

634
